# 第 5 章 练习题答案

> 精选 3 道核心练习，巩固预训练理解。

## 练习 5.1：初始损失为什么 ≈ ln(vocab)？

**题目**：随机初始化的模型，初始 loss 大约是多少？为什么？

In [ ]:
import torch
import torch.nn.functional as F
import math
from src.gpt import GPTModel, GPT_CONFIG_124M

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 64, "n_layers": 1, "n_heads": 4, "context_length": 8})
torch.manual_seed(0)
model = GPTModel(cfg)

# 随机输入和目标
x = torch.randint(0, cfg["vocab_size"], (4, 8))
y = torch.randint(0, cfg["vocab_size"], (4, 8))

with torch.no_grad():
    logits = model(x)
    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())

print(f"词表大小: {cfg['vocab_size']}")
print(f"ln(vocab) = ln({cfg['vocab_size']}) = {math.log(cfg['vocab_size']):.2f}")
print(f"实际初始 loss: {loss.item():.2f}")
print(f"\n💡 随机初始化时，模型对每个位置均匀预测所有 token，")
print(f"   交叉熵损失 = -log(1/vocab) = ln(vocab) ≈ {math.log(cfg['vocab_size']):.2f}")
print(f"   随着训练，loss 从这个值逐渐下降。")

## 练习 5.2：温度采样的效果

**题目**：对比不同 temperature 下，采样的概率分布有什么变化？

In [ ]:
import torch
import torch.nn.functional as F

# 假设某 token 的 logits
logits = torch.tensor([1.0, 2.0, 3.0, 0.5])

print(f"logits: {logits.tolist()}")
print(f"\n{'temperature':<14} {'概率分布':<40}")
print("-" * 54)
for T in [0.5, 1.0, 2.0, 5.0]:
    probs = F.softmax(logits / T, dim=-1)
    bar = " ".join(f"{p:.2f}" for p in probs)
    print(f"T={T:<11} [{bar}]")
print("\n💡 T→0：分布变尖锐（趋于 argmax，更确定）")
print("   T→∞：分布变均匀（更随机多样）")
print("   T=1：标准 softmax。生成常用 T=0.7~1.0。")

## 练习 5.3：权重加载为何要转置？

**题目**：加载 OpenAI GPT-2 权重时，为什么要对某些权重做转置？

**答案（概念题）**：OpenAI 用 TensorFlow 保存权重，其 `tf.layers.Dense` 的权重布局是 `[out, in]`，而 PyTorch 的 `nn.Linear.weight` 是 `[out, in]` 但前向计算用 `x @ W.T`。两者约定不完全一致，部分权重（如注意力的 qkv）需要转置对齐。

In [ ]:
# 演示：同一个矩阵运算，不同存储约定的差异
import torch

torch.manual_seed(0)
x = torch.randn(2, 3)       # 输入 [batch, in_features]
W_pytorch = torch.randn(3, 4)  # PyTorch: [in, out]，前向 x @ W

# PyTorch 风格
out_pytorch = x @ W_pytorch

# TensorFlow 风格（权重存为 [out, in]，前向 x @ W.T）
W_tf = W_pytorch.T          # 转置存储
out_tf = x @ W_tf.T         # 前向时再转置回来

print(f"两种方式结果相同: {torch.allclose(out_pytorch, out_tf)}")
print("\n💡 加载 OpenAI 权重时，要根据目标层的约定决定是否转置。")
print("   这就是 ch05/solution.py 里 load_weights 要处理键名+形状的原因。")